# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading and exploratory analysis of the FAIR^2 dataset using the `mlcroissant` library. You will learn to access Croissant schema metadata, review the available record sets and fields, extract tabular data, and perform basic filtering and normalization.

### Dataset Source
The dataset is defined by a Croissant schema available at:

**URL:** [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Install mlcroissant if not present
!pip install -q mlcroissant


## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"Published: {metadata.datePublished}\n\nVersion: {metadata.version}\n\nLicense: {metadata.license}")


## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In Croissant, each table or set of records is a *RecordSet* (with its own `@id`), and fields/columns have distinct `@id`s as well. We'll enumerate these.

In [ ]:
# List all record sets and their fields
record_sets = dataset.record_sets
print("FAIR^2 Record Sets detected:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields/Columns:")
        for field in fields:
            # Display field @id and name
            print(f"    - @id: {field['@id']} | name: {field.get('name', '(no name)')}")
    else:
        print("  No fields listed.")


**Preview sample records from each available RecordSet using their `@id`.**


In [ ]:
# Show sample record for each record set
for rs in record_sets:
    rs_id = rs["@id"]
    print(f"\nSample records for RecordSet @id: {rs_id}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print(f"No records loaded or error: {e}")


## 3. Data Extraction

Load all RecordSets into DataFrames for analysis.
Each RecordSet is referenced by its `@id`, and columns by their `@id`.

In [ ]:
# Build list of RecordSet @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nLoaded RecordSet: {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"RecordSet {rs_id}: No records found.")
    except Exception as e:
        print(f"RecordSet {rs_id}: Load error: {e}")


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, grouping. All operations reference columns by their `@id`.

- Select a numeric column for distribution analysis.
- Filter out records below a threshold.
- Normalize values.
- Optionally, group by a categorical column.


In [ ]:
# Choose the first RecordSet with numeric columns
selected_rs_id = None
numeric_field_id = None
group_field_id = None

# Find numeric column candidates
for rs_id, df in dataframes.items():
    numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if numeric_cols:
        selected_rs_id = rs_id
        numeric_field_id = numeric_cols[0]
        # Optionally pick a group field, e.g. categorical
        group_candidates = [col for col in df.columns if df[col].dtype == object]
        if group_candidates:
            group_field_id = group_candidates[0]
        break

if selected_rs_id and numeric_field_id:
    print(f"\nSelected RecordSet: {selected_rs_id}")
    print(f"Numeric field @id chosen for analysis: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()  # e.g., filter above mean
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records in {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric fields detected in loaded record sets.")


## 5. Visualization

Visualize numeric data distributions and relationships between fields. All axes/labels reference column `@id`s.


In [ ]:
# Basic plotting
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=7, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {selected_rs_id}")
    plt.show()

    # Visualize grouped mean if grouping available
    if group_field_id:
        plt.figure(figsize=(8,6))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Grouped Mean {numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion

This notebook showed how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` package. Key steps included reviewing schema metadata, enumerating RecordSets and their fields (referenced by `@id`), loading record data into DataFrames, filtering and normalizing numeric fields, grouping by categorical values, and visualizing distributions.

The FAIR^2 dataset, though small, is highly structured and provides a clear model for integrating clinical, laboratory, imaging, and genetic information in rare disease studies. All entity references were by their Croissant `@id`, supporting reproducibility and interoperability.


Feel free to extend this analysis with domain-specific statistical modeling, literature integration, or advanced clinical groupings using the Croissant schema structure.